# Dry run

In [1]:
import json, random
import numpy as np, pandas as pd
from pathlib import Path
from tokenizers import Tokenizer
from gpt.splits import BLOCK, parse_turns, split_of_block
from gpt.improv_data import build_improv_examples, split_turn

DATA = Path("../data_sft_blocks")
tok = Tokenizer.from_file(str(DATA / "tokenizer.json"))
eot_id, eoturn_id = tok.token_to_id("<|endoftext|>"), tok.token_to_id("<|endofturn|>")
shake = json.load(open(DATA / "meta.json"))
print("ids:", eot_id, eoturn_id, "| vocab:", tok.get_vocab_size())
assert eoturn_id is not None and tok.get_vocab_size() == shake["vocab_size"]
print("marker overhead (tokens):", len(tok.encode("[USER] ")), len(tok.encode("[ASSISTANT] ")))

turns = parse_turns((DATA / "raw/input.txt").read_text(encoding="utf-8"))
assert len(turns) == shake["turns"], "turn count differs from the Shakespeare SFT data"
train, val, stats = build_improv_examples(turns, tok, eot_id=eot_id, eoturn_id=eoturn_id, seed=42, max_reply_tokens=60, role_markers=False)
print(json.dumps(stats, indent=1))
assert train and val, "an empty split"

val_blocks = set(shake["val_blocks"])
for name, exs in (("train", train), ("val", val)):
    for e in exs:
        assert e["ids"][0] == eot_id and e["ids"][-1] == eoturn_id
        assert e["total_len"] <= 256 and e["total_len"] - e["prompt_len"] >= 2
        assert all(i // BLOCK == e["block_index"] for i in e["context_indices"] + [e["target_index"]]), "context crosses a block"
        assert split_of_block(e["block_index"]) == name and (e["block_index"] in val_blocks) == (name == "val")
print("invariants OK")

sample = random.Random(0).sample(train, min(500, len(train)))
bad = sum(tok.encode(tok.decode(e["ids"][1:-1])).ids != e["ids"][1:-1] for e in sample)
print(f"separate vs joint tokenization mismatches: {bad} / {len(sample)}")


def summarize(exs, name):
    df = pd.DataFrame({
        "reply_tokens": [e["total_len"] - e["prompt_len"] - 1 for e in exs],
        "total_len": [e["total_len"] for e in exs],
        "n_context": [len(e["context_indices"]) for e in exs],
        "user": [e["user_speaker"] for e in exs],
        "target": [split_turn(turns[e["target_index"]])[0] for e in exs],
        "episode": [(e["block_index"], e["episode_index"]) for e in exs],
    })
    print(f"\n=== {name}: {len(df)} examples | {df.episode.nunique()} episodes | {df.target.nunique()} target speakers")
    print(df[["reply_tokens", "total_len"]].describe().round(1).T)
    print("replies <=3 tokens:", f"{(df.reply_tokens <= 3).mean():.1%}", "| context lengths:", df.n_context.value_counts().sort_index().to_dict())
    print("top targets:", df.target.value_counts(normalize=True).head(6).round(3).to_dict())
    print("top users:  ", df.user.value_counts(normalize=True).head(6).round(3).to_dict())

summarize(train, "train"); summarize(val, "val")

random.seed(1)
for e in random.sample(train, 8):
    print("-" * 70, f"\nuser={e['user_speaker']} | context turns={len(e['context_indices'])}")
    print(tok.decode(e["ids"][1:e["prompt_len"]]))
    print(">>>", tok.decode(e["ids"][e["prompt_len"]:-1]))

ids: 0 1 | vocab: 4096
marker overhead (tokens): 7 12
{
 "train_count": 1630,
 "val_count": 158,
 "rejections": {
  "target_is_user": 1771,
  "reply_too_long": 626,
  "total_too_long": 297,
  "user_missing_from_context": 2047,
  "empty_reply": 47,
  "no_eligible_user": 18
 }
}
invariants OK
separate vs joint tokenization mismatches: 0 / 500

=== train: 1630 examples | 514 episodes | 209 target speakers
               count   mean   std   min   25%    50%    75%    max
reply_tokens  1630.0   21.7  14.6   2.0  10.0   16.0   30.0   60.0
total_len     1630.0  124.9  56.0  26.0  80.2  115.0  164.8  256.0
replies <=3 tokens: 1.3% | context lengths: {1: 487, 2: 559, 3: 584}
top targets: {'GLOUCESTER:': 0.031, 'KING RICHARD III:': 0.028, 'PETRUCHIO:': 0.026, 'DUKE VINCENTIO:': 0.025, 'MENENIUS:': 0.024, 'ROMEO:': 0.021}
top users:   {'GLOUCESTER:': 0.038, 'ROMEO:': 0.028, 'LUCIO:': 0.027, 'PETRUCHIO:': 0.025, 'DUKE VINCENTIO:': 0.023, 'CORIOLANUS:': 0.021}

=== val: 158 examples | 52 episodes 

In [2]:
import json, random
rows = [json.loads(l) for l in open("../data_improv/train_examples.jsonl")]
random.seed(3)
for r in random.sample(rows, 6):
    print("-" * 70, f"\nuser = {r['user_speaker']}  | user line: {r['user_line']!r}")
    print("\n\n".join(r["context_turns"]))
    print(">>>", r["target_turn"])

---------------------------------------------------------------------- 
user = QUEEN ELIZABETH:  | user line: "'Tis full of thy foul wrongs."
KING RICHARD III:
Now, by the world--

QUEEN ELIZABETH:
'Tis full of thy foul wrongs.
>>> KING RICHARD III:
My father's death--
---------------------------------------------------------------------- 
user = CLAUDIO:  | user line: 'O heavens! it cannot be.'
ISABELLA:
O, 'tis the cunning livery of hell,
The damned'st body to invest and cover
In prenzie guards! Dost thou think, Claudio?
If I would yield him my virginity,
Thou mightst be freed.

CLAUDIO:
O heavens! it cannot be.
>>> ISABELLA:
Yes, he would give't thee, from this rank offence,
So to offend him still. This night's the time
That I should do what I abhor to name,
Or else thou diest to-morrow.
---------------------------------------------------------------------- 
user = PAULINA:  | user line: "Good my lord, forbear:\nThe ruddiness upon her lip is wet;\nYou'll mar it if you kiss it, stain

## From here

In [1]:
%load_ext autoreload
%autoreload 2
import json, torch, pandas as pd
import torch.nn.functional as F
from gpt.generate import load_model
from gpt.improv_data import get_val_example_batches
from gpt.config import Config
print("copy_embedding" in Config.__dataclass_fields__)      # must be True

DATA = "../data_improv"
eoturn_id = json.load(open(f"{DATA}/meta.json"))["eoturn_id"]


@torch.no_grad()
def decomposed_val(path, B=16):
    model, cfg = load_model(path)
    tot = {"content": [0.0, 0], "stop": [0.0, 0]}
    for x, y in get_val_example_batches(DATA, B, "cpu"):
        logits, _ = model(x)
        nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1),
                              ignore_index=-100, reduction="none").reshape(y.shape)
        valid = y != -100
        stop = valid & (y == eoturn_id)
        for k, m in (("content", valid & ~stop), ("stop", stop)):
            tot[k][0] += nll[m].sum().item()
            tot[k][1] += int(m.sum())
    c, s = tot["content"][0] / tot["content"][1], tot["stop"][0] / tot["stop"][1]
    n = tot["content"][1] + tot["stop"][1]
    return {"overall": (tot["content"][0] + tot["stop"][0]) / n, "content_nll": c, "stop_nll": s,
            "p_stop_at_end": float(torch.exp(torch.tensor(-s))),
            "stop_share_of_loss": tot["stop"][0] / (tot["content"][0] + tot["stop"][0])}


runs = {"pretrain_v2": "../runs/pretrain_v2/best.pt",
        "stage1 (sft_blocks)": "../runs/sft_blocks_lr1e-4/best.pt",
        "improv_from_blocks": "../runs/improv_from_blocks/best.pt",
        "improv_from_pretrain": "../runs/improv_from_pretrain/best.pt",
        "improv_warm": "../runs/improv_warm/best.pt"}
print(pd.DataFrame({k: decomposed_val(v) for k, v in runs.items()}).T.round(3).to_string())

True
                      overall  content_nll  stop_nll  p_stop_at_end  stop_share_of_loss
pretrain_v2             6.508        5.999    16.634          0.000               0.122
stage1 (sft_blocks)     4.453        3.663    20.185          0.000               0.217
improv_from_blocks      3.672        3.829     0.534          0.586               0.007
improv_from_pretrain    4.086        4.272     0.382          0.682               0.004
improv_warm             3.556        3.709     0.522          0.593               0.007


In [2]:
%load_ext autoreload
%autoreload 2
import re, json, numpy as np, pandas as pd, torch
from collections import Counter
from pathlib import Path
from tokenizers import Tokenizer
from gpt.generate import load_model, generate_batch
from gpt.utils import check_tokenizer, checkpoint_tokenizer_hash

# RUN, DATA = "../runs/improv_from_blocks", "../data_improv"
# TAG, MAX_NEW, N = "stage2_improv_from_blocks", 60, 8
RUN, DATA = "../runs/improv_warm", "../data_improv"
TAG, MAX_NEW, N = "stage2_improv_warm", 60, 8
tok = Tokenizer.from_file(f"{DATA}/tokenizer.json")
model, cfg = load_model(f"{RUN}/best.pt")
check_tokenizer(f"{DATA}/tokenizer.json", cfg.tokenizer_sha256 or checkpoint_tokenizer_hash(f"{RUN}/best.pt"), "checkpoint")
meta = json.load(open(f"{DATA}/meta.json"))
eot_id, eoturn_id = meta["eot_id"], meta["eoturn_id"]
assert meta["role_markers"] is False, "this cell builds prompts without [USER]/[ASSISTANT] markers"

# ---- words ----
words = lambda s: [w.strip("'") for w in re.findall(r"[a-z']+", s.lower()) if w.strip("'")]
STOPW = {"the", "a", "an", "and", "or", "but", "to", "of", "in", "on", "at", "is", "are", "was", "i", "you", "he",
         "she", "it", "my", "thy", "thou", "thee", "that", "this", "with", "for", "not", "have", "hast", "will"}
content = lambda s: {w for w in words(s) if w not in STOPW and len(w) > 2}
SHAKE = set(words(open("../data_sft_blocks/raw/input.txt", encoding="utf-8").read()))
sv = Path("../data_sft_blocks/story_vocab.json")
assert sv.exists(), "run the stage-1 baseline cell first: it builds story_vocab.json"
STORY = set(json.load(open(sv)))
word_class = lambda w: "shake" if w in SHAKE else ("story" if w in STORY else "invented")


def class_rates(ws):
    cs = [word_class(w) for w in ws] or ["invented"]
    return {c: cs.count(c) / len(cs) for c in ("shake", "story", "invented")}


def repeat_bigram(w):
    return 0.0 if len(w) < 2 else 1 - len(set(zip(w, w[1:]))) / (len(w) - 1)


def distinct_n(rs, n=2):
    g = [x for r in rs for x in zip(*[words(r)[i:] for i in range(n)])]
    return len(set(g)) / max(1, len(g))


# ---- contexts ----
HAND = [
    ([("ROMEO", "Wilt thou not walk with me beneath the moon?")], "JULIET"),
    ([("ROMEO", "Hey, I think I left my keys in your garden.")], "JULIET"),
    ([("PETRUCHIO", "Good morrow, sweet Kate! Hast thou seen my hat?")], "KATHARINA"),
    ([("LUCIO", "Come, sir, a jest! the duke is gone to sea.")], "ISABELLA"),
    ([("MENENIUS", "The people murmur, and the corn grows scarce.")], "SICINIUS"),
    ([("GLOUCESTER", "Our brother sleeps; let us not wake the court.")], "QUEEN ELIZABETH"),
    ([("ROMEO", "I have a gift for thee."), ("JULIET", "A gift? What manner of gift?"),
      ("ROMEO", "A boat, that we may sail away tonight.")], "JULIET"),
    ([("LEONTES", "Thou hast a dragon in thy cellar, friend!")], "PAULINA"),
]
hand_ctxs = [{"context_turns": [f"{s}:\n{l}" for s, l in h], "partner_tag": f"{p}:", "user_line": h[-1][1]} for h, p in HAND]
val_rows = [json.loads(l) for l in open(f"{DATA}/val_examples.jsonl")]
val_ctxs = [{"context_turns": r["context_turns"], "partner_tag": r["target_turn"].split("\n")[0], "user_line": r["user_line"]}
            for r in val_rows if r["prompt_len"] <= cfg.context_len - MAX_NEW]
print(len(hand_ctxs), "hand contexts |", len(val_ctxs), "val contexts")


@torch.no_grad()
def sample(ctx, temperature, k, seed=0):
    prompt = "\n\n".join(ctx["context_turns"]) + "\n\n" + ctx["partner_tag"] + "\n"
    idx = torch.tensor([[eot_id] + tok.encode(prompt).ids] * N)          # one context replicated N times: equal lengths
    g = torch.Generator().manual_seed(seed)
    out, finished, n_new = generate_batch(model, cfg.context_len, idx, MAX_NEW, temperature, k, g, [eoturn_id, eot_id], eot_id)
    T0, reps = idx.shape[1], []
    for b in range(N):
        new = out[b, T0:T0 + int(n_new[b])].tolist()
        end = new[-1] if (bool(finished[b]) and new) else None
        text = tok.decode(new[:-1] if end is not None else new).strip()
        reps.append({"reply": text,
                     "ended": "turn" if end == eoturn_id else ("doc" if end == eot_id else "truncated"),
                     "leak": bool(re.search(r"^[A-Z][A-Za-z' ]*:\s*$", text, flags=re.M))})   # invented a new speaker tag
    return reps


def ctx_metrics(ctx, reps):
    r = [x["reply"] for x in reps]
    w = [words(x) for x in r]
    cu = content(ctx["user_line"])
    rates = [class_rates(x) for x in w]
    out = {
        "ended_turn": float(np.mean([x["ended"] == "turn" for x in reps])),
        "truncated": float(np.mean([x["ended"] == "truncated" for x in reps])),
        "leak": float(np.mean([x["leak"] for x in reps])),
        "words": float(np.mean([len(x) for x in w])),
        "overlap": float(np.mean([len(content(x) & cu) for x in r])),
        "overlap_frac": float(np.mean([len(content(x) & cu) / max(1, len(content(x))) for x in r])),
        "new_words": float(np.mean([len(content(x) - cu) for x in r])),
        "repeat_bigram": float(np.mean([repeat_bigram(x) for x in w])),
        "distinct2": distinct_n(r),
    }
    for c in ("shake", "story", "invented"):
        out[f"{c}_words"] = float(np.mean([q[c] for q in rates]))
    return out


SETTINGS = {"t10": (1.0, None), "t08k50": (0.8, 50)}
Path("../runs/improv").mkdir(parents=True, exist_ok=True)
results = {}
for name, (temperature, k) in SETTINGS.items():
    for set_name, ctxs in (("hand", hand_ctxs), ("val", val_ctxs)):
        rows = []
        for i, ctx in enumerate(ctxs):
            reps = sample(ctx, temperature, k, seed=i)
            rows.append({"ctx": ctx, "replies": reps, "metrics": ctx_metrics(ctx, reps)})
        mean = {key: float(np.mean([r["metrics"][key] for r in rows])) for key in rows[0]["metrics"]}
        results[(set_name, name)] = (rows, mean)
        json.dump({"run": RUN, "set": set_name, "sampling": {"temperature": temperature, "k": k, "n": N, "max_new_tokens": MAX_NEW},
                   "mean": mean, "rows": rows}, open(f"../runs/improv/{TAG}_{set_name}_{name}.json", "w"), indent=1)

print(pd.DataFrame({f"{s}/{n}": m for (s, n), (_, m) in results.items()}).T.round(3).to_string())

for name in SETTINGS:                                                 # read the text yourself
    rows = results[("hand", name)][0]
    print("=" * 30, name)
    for row in rows[:4]:
        print(row["ctx"]["partner_tag"], "<-", row["ctx"]["user_line"])
        for x in row["replies"][:3]:
            print("  •", x["ended"], "|", x["reply"].replace("\n", " / ") or "(empty)")
    inv = [w for r in rows for x in r["replies"] for w in words(x["reply"]) if word_class(w) == "invented"]
    print("invented:", sorted(set(inv))[:30])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
8 hand contexts | 150 val contexts
             ended_turn  truncated   leak   words  overlap  overlap_frac  new_words  repeat_bigram  distinct2  shake_words  story_words  invented_words
hand/t10          0.609      0.094  0.031  12.359    0.078         0.014      7.125          0.002      0.992        0.910        0.002           0.088
val/t10           0.640      0.035  0.022  10.362    0.182         0.032      5.748          0.002      0.993        0.914        0.005           0.080
hand/t08k50       0.641      0.016  0.000   9.234    0.203         0.059      4.250          0.001      0.982        0.975        0.003           0.023
val/t08k50        0.628      0.016  0.015   9.859    0.224         0.053      4.275          0.010      0.963        0.979        0.002           0.019
============================== t10
JULIET: <- Wilt thou not walk with me beneath the moon?
  • turn | Or shall you re

In [3]:
import json, numpy as np, pandas as pd

A, B = "stage2_improv_from_blocks", "stage2_improv_warm"
load = lambda tag, d: json.load(open(f"../runs/improv/{tag}_val_{d}.json"))["rows"]
keys = ["ended_turn", "truncated", "leak", "words", "overlap", "overlap_frac", "new_words", "distinct2", "invented_words"]
out = {}
for d in ("t10", "t08k50"):
    ra, rb = load(A, d), load(B, d)
    assert len(ra) == len(rb)
    diffs = {k: np.array([b["metrics"][k] - a["metrics"][k] for a, b in zip(ra, rb)]) for k in keys}
    out[d] = {k: f"{v.mean():+.3f} ± {v.std(ddof=1) / np.sqrt(len(v)):.3f}" for k, v in diffs.items()}
print(pd.DataFrame(out).to_string())

                           t10          t08k50
ended_turn      -0.280 ± 0.016  -0.349 ± 0.014
truncated       -0.037 ± 0.010  -0.003 ± 0.005
leak            -0.003 ± 0.006  -0.010 ± 0.005
words           -2.607 ± 0.314  -0.502 ± 0.297
overlap         -0.065 ± 0.018  -0.047 ± 0.018
overlap_frac    -0.002 ± 0.004  -0.004 ± 0.005
new_words       -1.718 ± 0.177  -0.556 ± 0.134
distinct2       +0.003 ± 0.001  +0.004 ± 0.003
invented_words  -0.007 ± 0.005  -0.006 ± 0.002
